# Neuron Interaction

#### Usando la matrix W para evaluar diferentes interacciones

## Qué agrega esta notebook

Ahora las neuronas dejan de ser independientes. Un spike de una neurona puede modificar la corriente de otra mediante una matriz de conectividad `W`.

```text
input externo ───────────────┐
ruido individual ────────────┼→ corriente total → dinámica neuronal → spikes
spikes de otras neuronas → W ┘                                  │
                          ↑_______________________________________┘
```

La realimentación aparece porque los spikes actualizan estados sinápticos que vuelven a entrar como corriente en pasos posteriores.

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

import ipywidgets as widgets
from IPython.display import display, clear_output

In [12]:
# ============================================================
# 1. PARÁMETROS PREDEFINIDOS DEL MODELO DE IZHIKEVICH
# ============================================================

NEURON_PRESETS = {
    "Regular Spiking (RS)": {
        "a": 0.02,
        "b": 0.20,
        "c": -65.0,
        "d": 8.0,
        "description": "Disparo regular con cierta adaptación. Buen punto de partida para neuronas excitatorias o motoneuronas idealizadas."
    },
    "Intrinsically Bursting (IB)": {
        "a": 0.02,
        "b": 0.20,
        "c": -55.0,
        "d": 4.0,
        "description": "Tiende a producir ráfagas iniciales de spikes. Útil para explorar bursting."
    },
    "Chattering (CH)": {
        "a": 0.02,
        "b": 0.20,
        "c": -50.0,
        "d": 2.0,
        "description": "Produce ráfagas rápidas y repetidas. Útil para estudiar actividad oscilatoria intensa."
    },
    "Fast Spiking (FS)": {
        "a": 0.10,
        "b": 0.20,
        "c": -65.0,
        "d": 2.0,
        "description": "Disparo rápido con poca adaptación. Suele usarse como aproximación para interneuronas inhibitorias."
    },
    "Low Threshold Spiking (LTS)": {
        "a": 0.02,
        "b": 0.25,
        "c": -65.0,
        "d": 2.0,
        "description": "Puede responder con umbrales más bajos. Útil para explorar interneuronas o neuronas más sensibles."
    }
}

In [13]:
# ============================================================
# 2. GENERADOR DE INPUTS
# ============================================================

def generate_input(
    input_type="motor_plan",
    t=None,
    amplitude=10.0,
    start=100.0,
    end=900.0,
    frequency=5.0
):
    """
    Genera el input externo común I_externo(t).

    Este input todavía no es un dato real.
    Es un comando artificial para explorar la respuesta de la red.
    """

    I = np.zeros_like(t)
    active = (t >= start) & (t <= end)

    if input_type == "constant":
        I[active] = amplitude

    elif input_type == "pulse":
        pulse_end = start + 100
        active_pulse = (t >= start) & (t <= pulse_end)
        I[active_pulse] = amplitude

    elif input_type == "ramp":
        duration = max(end - start, 1)
        I[active] = amplitude * (t[active] - start) / duration

    elif input_type == "sinusoidal":
        I[active] = amplitude * (
            0.5 + 0.5 * np.sin(2 * np.pi * frequency * (t[active] - start) / 1000)
        )

    elif input_type == "motor_plan":
        # Reposo -> subida -> mantenimiento -> relajación
        rise_start = start
        rise_end = start + 200

        hold_start = rise_end
        hold_end = end - 200

        fall_start = hold_end
        fall_end = end

        rise = (t >= rise_start) & (t < rise_end)
        hold = (t >= hold_start) & (t < hold_end)
        fall = (t >= fall_start) & (t <= fall_end)

        if rise_end > rise_start:
            I[rise] = amplitude * (t[rise] - rise_start) / (rise_end - rise_start)

        I[hold] = amplitude

        if fall_end > fall_start:
            I[fall] = amplitude * (1 - (t[fall] - fall_start) / (fall_end - fall_start))

    return I

In [14]:
# ============================================================
# 3. CREACIÓN DE MATRIZ DE CONECTIVIDAD W
# ============================================================

def create_connectivity_matrix(
    n_neurons=30,
    connection_probability=0.15,
    weight_scale=0.5,
    connection_type="excitatory",
    random_seed=42
):
    """
    Crea una matriz W de conectividad sináptica.

    W[i, j] representa el efecto de la neurona i sobre la neurona j.

    Si W[i, j] > 0: conexión excitatoria.
    Si W[i, j] < 0: conexión inhibitoria.
    Si W[i, j] = 0: no hay conexión.

    En esta primera versión:
    - las conexiones son aleatorias;
    - no hay autoconexiones;
    - el tamaño de los pesos se controla con weight_scale.
    """

    rng = np.random.default_rng(random_seed)

    # Máscara de conexiones aleatorias
    mask = rng.random((n_neurons, n_neurons)) < connection_probability

    # Sin autoconexiones
    np.fill_diagonal(mask, False)

    # Pesos aleatorios pequeños
    W = weight_scale * rng.random((n_neurons, n_neurons)) * mask

    if connection_type == "excitatory":
        W = np.abs(W)

    elif connection_type == "inhibitory":
        W = -np.abs(W)

    elif connection_type == "mixed":
        # 80% excitatorias, 20% inhibitorias aprox.
        signs = rng.choice([1, -1], size=(n_neurons, n_neurons), p=[0.8, 0.2])
        W = W * signs

    return W

### Cómo leer la matriz `W`

La convención de esta notebook es:

```text
W[i, j] = efecto de la neurona origen i sobre la neurona destino j
```

Ejemplo para tres neuronas:

| Origen \ Destino | N1 | N2 | N3 |
|---|---:|---:|---:|
| N1 | 0 | +0.5 | 0 |
| N2 | 0 | 0 | −0.8 |
| N3 | +0.3 | 0 | 0 |

- `W[0,1]=+0.5`: un estado sináptico de N1 excita a N2.
- `W[1,2]=−0.8`: N2 inhibe a N3.
- `0`: no existe esa conexión.
- La diagonal se anula para evitar autoconexiones.

La matriz no contiene spikes ni corrientes temporales: contiene **la arquitectura y magnitud de las conexiones**.

In [15]:
# ============================================================
# 4. SIMULACIÓN DE RED CONECTADA
# ============================================================

def simulate_connected_izhikevich_network(
    n_neurons=30,
    neuron_type="Regular Spiking (RS)",
    total_time=1000,
    dt=0.5,
    input_type="motor_plan",
    amplitude=10.0,
    input_start=100,
    input_end=900,
    frequency=5.0,
    parameter_noise=0.05,
    input_noise=1.0,
    connection_probability=0.15,
    weight_scale=0.5,
    connection_type="excitatory",
    synaptic_tau=10.0,
    random_seed=42
):
    """
    Simula una red de neuronas Izhikevich conectadas por una matriz W.

    Input total por neurona:

        I_total_j(t) = I_externo(t) + ruido_j(t) + I_sinaptico_j(t)

    donde:

        I_sinaptico_j(t) = sum_i W[i, j] * s_i(t)

    y s_i(t) es una variable sináptica que:
    - aumenta cuando la neurona i dispara;
    - decae exponencialmente con el tiempo.
    """

    rng = np.random.default_rng(random_seed)

    t = np.arange(0, total_time, dt)
    n_steps = len(t)

    # Parámetros base según tipo neuronal seleccionado
    preset = NEURON_PRESETS[neuron_type]

    base_a = preset["a"]
    base_b = preset["b"]
    base_c = preset["c"]
    base_d = preset["d"]

    # Variación individual de parámetros
    a = base_a * (1 + parameter_noise * rng.normal(size=n_neurons))
    b = base_b * (1 + parameter_noise * rng.normal(size=n_neurons))
    c = base_c + abs(base_c) * parameter_noise * rng.normal(size=n_neurons)
    d = base_d * (1 + parameter_noise * rng.normal(size=n_neurons))

    a = np.clip(a, 0.001, 0.2)
    b = np.clip(b, 0.01, 0.4)
    c = np.clip(c, -90, -40)
    d = np.clip(d, 0.1, 20)

    params = {
        "a": a,
        "b": b,
        "c": c,
        "d": d
    }

    # Input externo común
    I_external = generate_input(
        input_type=input_type,
        t=t,
        amplitude=amplitude,
        start=input_start,
        end=input_end,
        frequency=frequency
    )

    # Matriz de conectividad
    W = create_connectivity_matrix(
        n_neurons=n_neurons,
        connection_probability=connection_probability,
        weight_scale=weight_scale,
        connection_type=connection_type,
        random_seed=random_seed
    )

    # Estados
    V = np.zeros((n_neurons, n_steps))
    U = np.zeros((n_neurons, n_steps))
    spikes = np.zeros((n_neurons, n_steps))

    # Variable sináptica por neurona
    S = np.zeros((n_neurons, n_steps))

    # Corriente sináptica recibida por cada neurona
    I_syn = np.zeros((n_neurons, n_steps))

    # Input total
    I_total = np.zeros((n_neurons, n_steps))

    # Condiciones iniciales
    V[:, 0] = -65.0
    U[:, 0] = b * V[:, 0]

    # Ruido individual de input
    I_noise = input_noise * rng.normal(size=(n_neurons, n_steps))

    # Factor de decaimiento sináptico
    syn_decay = np.exp(-dt / synaptic_tau)

    for step in range(1, n_steps):

        # La variable sináptica decae
        S[:, step] = S[:, step - 1] * syn_decay

        # Corriente sináptica: suma de conexiones desde otras neuronas
        # W[i, j] = efecto de i sobre j
        I_syn[:, step] = W.T @ S[:, step]

        # Input total recibido por cada neurona
        I_total[:, step] = I_external[step] + I_noise[:, step] + I_syn[:, step]

        # Dinámica de Izhikevich
        dv = 0.04 * V[:, step - 1]**2 + 5 * V[:, step - 1] + 140 - U[:, step - 1] + I_total[:, step]
        du = a * (b * V[:, step - 1] - U[:, step - 1])

        V[:, step] = V[:, step - 1] + dt * dv
        U[:, step] = U[:, step - 1] + dt * du

        fired = V[:, step] >= 30

        # Mostrar spike en gráfico
        V[fired, step - 1] = 30

        # Reset
        V[fired, step] = c[fired]
        U[fired, step] = U[fired, step] + d[fired]

        # Registro de spike
        spikes[fired, step] = 1

        # Cada spike incrementa la variable sináptica de esa neurona
        S[fired, step] += 1

    return {
        "t": t,
        "V": V,
        "U": U,
        "spikes": spikes,
        "S": S,
        "I_external": I_external,
        "I_syn": I_syn,
        "I_total": I_total,
        "W": W,
        "params": params,
        "neuron_description": preset["description"]
    }

### Qué ocurre en cada paso temporal

1. La variable sináptica `S` decae exponencialmente.
2. Se calcula `I_syn = W.T @ S`. La transpuesta convierte estados de origen en corrientes de destino.
3. Se suman input externo, ruido y corriente sináptica.
4. Euler actualiza `v` y `u`.
5. Las neuronas que alcanzan 30 mV registran un spike y se resetean.
6. Cada spike incrementa su estado `S`, que influirá sobre la red.

`synaptic_tau` controla cuánto dura ese efecto. Una constante mayor hace que la influencia de un spike desaparezca más lentamente.

La semilla fija simultáneamente parámetros, conectividad y ruido. Por eso dos configuraciones comparadas con la misma semilla comparten la misma realización aleatoria.

In [16]:
# ============================================================
# 5. MÉTRICAS SIMPLES
# ============================================================

def compute_population_activity(spikes, dt=0.5, bin_size_ms=20):
    """
    Cuenta spikes en ventanas temporales.
    """

    n_neurons, n_steps = spikes.shape
    steps_per_bin = max(int(bin_size_ms / dt), 1)
    n_bins = n_steps // steps_per_bin

    activity = np.zeros(n_bins)
    time_bins = np.zeros(n_bins)

    for i in range(n_bins):
        start = i * steps_per_bin
        end = start + steps_per_bin

        activity[i] = spikes[:, start:end].sum()
        time_bins[i] = start * dt

    return time_bins, activity

### Interpretación y límites

- La **corriente sináptica media** puede ocultar que algunas neuronas reciben excitación y otras inhibición; conviene leerla junto con `W`.
- Un aumento de actividad poblacional no implica necesariamente sincronización: el raster muestra si los spikes coinciden temporalmente.
- La probabilidad de conexión controla densidad; `weight_scale` controla magnitud. Son conceptos distintos.
- Esta red usa sinapsis de corriente simplificadas, sin potenciales de reversión, retardos axonales ni plasticidad.

In [17]:
# ============================================================
# 6. VISUALIZACIÓN DE MATRIZ W
# ============================================================

def plot_connectivity_matrix(W):
    """
    Muestra la matriz W como heatmap.
    """

    plt.figure(figsize=(8, 7))

    vmax = np.max(np.abs(W)) if np.max(np.abs(W)) > 0 else 1

    im = plt.imshow(
        W,
        cmap="coolwarm",
        aspect="auto",
        vmin=-vmax,
        vmax=vmax
    )

    plt.title("Matriz de conectividad W")
    plt.xlabel("Neurona destino j")
    plt.ylabel("Neurona origen i")

    cbar = plt.colorbar(im)
    cbar.set_label("Peso sináptico W[i, j]")

    plt.tight_layout()
    plt.show()

In [18]:
# ============================================================
# 7. VISUALIZACIÓN DEL GRAFO DE CONECTIVIDAD
# ============================================================

def plot_connectivity_graph(W, show_labels=True):
    """
    Muestra la red como grafo dirigido.

    Cada nodo es una neurona.
    Cada flecha representa una conexión W[i, j] distinta de cero.
    """

    n_neurons = W.shape[0]

    G = nx.DiGraph()

    for i in range(n_neurons):
        G.add_node(f"N{i+1}")

    for i in range(n_neurons):
        for j in range(n_neurons):
            if W[i, j] != 0:
                G.add_edge(f"N{i+1}", f"N{j+1}", weight=W[i, j])

    pos = nx.circular_layout(G)

    plt.figure(figsize=(9, 9))

    weights = np.array([G[u][v]["weight"] for u, v in G.edges()])

    if len(weights) > 0:
        widths = 0.5 + 3 * np.abs(weights) / np.max(np.abs(weights))
    else:
        widths = 1

    positive_edges = [(u, v) for u, v in G.edges() if G[u][v]["weight"] > 0]
    negative_edges = [(u, v) for u, v in G.edges() if G[u][v]["weight"] < 0]

    nx.draw_networkx_nodes(
        G,
        pos,
        node_size=700 if n_neurons <= 20 else 400,
        node_color="white",
        edgecolors="black"
    )

    nx.draw_networkx_edges(
        G,
        pos,
        edgelist=positive_edges,
        arrows=True,
        arrowstyle="->",
        arrowsize=12,
        width=1.0,
        alpha=0.5
    )

    nx.draw_networkx_edges(
        G,
        pos,
        edgelist=negative_edges,
        arrows=True,
        arrowstyle="->",
        arrowsize=12,
        width=1.0,
        alpha=0.5,
        style="dashed"
    )

    if show_labels:
        labels = {f"N{i+1}": f"N{i+1}" for i in range(n_neurons)}
        nx.draw_networkx_labels(
            G,
            pos,
            labels=labels,
            font_size=8 if n_neurons <= 20 else 6
        )

    plt.title("Grafo de conectividad neuronal")
    plt.axis("off")
    plt.show()

In [19]:
# ============================================================
# 8. VISUALIZACIÓN GENERAL DE RESULTADOS
# ============================================================

def plot_network_results(results, bin_size_ms=20, n_voltage_traces=5):
    """
    Grafica resultados principales de la red.
    """

    t = results["t"]
    V = results["V"]
    spikes = results["spikes"]
    I_external = results["I_external"]
    I_syn = results["I_syn"]

    n_neurons, n_steps = V.shape

    time_bins, activity = compute_population_activity(
        spikes=spikes,
        dt=t[1] - t[0],
        bin_size_ms=bin_size_ms
    )

    fig, axes = plt.subplots(6, 1, figsize=(15, 22), sharex=False)

    # 1. Input externo
    axes[0].plot(t, I_external)
    axes[0].set_title("Input externo común")
    axes[0].set_ylabel("I externo")
    axes[0].set_xlabel("Tiempo (ms)")
    axes[0].grid(True)

    # 2. Corriente sináptica media
    axes[1].plot(t, I_syn.mean(axis=0))
    axes[1].set_title("Corriente sináptica media recibida por la red")
    axes[1].set_ylabel("I sináptica media")
    axes[1].set_xlabel("Tiempo (ms)")
    axes[1].grid(True)

    # 3. Voltaje de algunas neuronas
    selected_neurons = np.linspace(
        0,
        n_neurons - 1,
        min(n_voltage_traces, n_neurons),
        dtype=int
    )

    for idx in selected_neurons:
        axes[2].plot(t, V[idx], label=f"N{idx+1}")

    axes[2].set_title("Potencial de membrana de algunas neuronas")
    axes[2].set_ylabel("v(t)")
    axes[2].set_xlabel("Tiempo (ms)")
    axes[2].legend()
    axes[2].grid(True)

    # 4. Raster plot
    for neuron_idx in range(n_neurons):
        spike_times = t[spikes[neuron_idx] == 1]
        axes[3].vlines(
            spike_times,
            neuron_idx + 0.5,
            neuron_idx + 1.5,
            linewidth=0.8
        )

    axes[3].set_title("Raster plot")
    axes[3].set_ylabel("Neurona")
    axes[3].set_xlabel("Tiempo (ms)")
    axes[3].set_ylim(0.5, n_neurons + 0.5)
    axes[3].grid(True)

    # 5. Heatmap de voltaje
    im = axes[4].imshow(
        V,
        aspect="auto",
        origin="lower",
        extent=[t[0], t[-1], 1, n_neurons],
        interpolation="nearest"
    )

    axes[4].set_title("Mapa de calor del potencial de membrana")
    axes[4].set_ylabel("Neurona")
    axes[4].set_xlabel("Tiempo (ms)")

    cbar = fig.colorbar(im, ax=axes[4])
    cbar.set_label("v(t)")

    # 6. Actividad poblacional
    axes[5].bar(
        time_bins,
        activity,
        width=bin_size_ms,
        align="edge"
    )

    axes[5].set_title(f"Actividad poblacional: spikes cada {bin_size_ms} ms")
    axes[5].set_ylabel("Cantidad de spikes")
    axes[5].set_xlabel("Tiempo (ms)")
    axes[5].grid(True)

    plt.tight_layout()
    plt.show()

In [20]:
# ============================================================
# 9. UTILIDAD PARA CREAR SLIDERS CON DESCRIPCIÓN
# ============================================================

def described_widget(description_text, widget):
    """
    Crea una fila con una descripción a la izquierda y el widget a la derecha.
    """

    description = widgets.HTML(
        value=f"<div style='width:260px; font-size:13px;'>{description_text}</div>"
    )

    return widgets.HBox([description, widget])

In [ ]:
# ============================================================
# 10. GUI COMPLETA
# ============================================================

output = widgets.Output()

# Widgets principales

neuron_type_widget = widgets.Dropdown(
    options=list(NEURON_PRESETS.keys()),
    value="Regular Spiking (RS)",
    description=""
)

n_neurons_widget = widgets.IntSlider(
    value=10,
    min=2,
    max=30,
    step=1,
    description=""
)

input_type_widget = widgets.Dropdown(
    options=["constant", "pulse", "ramp", "sinusoidal", "motor_plan"],
    value="motor_plan",
    description=""
)

amplitude_widget = widgets.FloatSlider(
    value=10.0,
    min=0.0,
    max=30.0,
    step=0.5,
    description=""
)

total_time_widget = widgets.IntSlider(
    value=1000,
    min=500,
    max=3000,
    step=100,
    description=""
)

dt_widget = widgets.FloatSlider(
    value=0.5,
    min=0.1,
    max=2.0,
    step=0.1,
    description=""
)

input_start_widget = widgets.IntSlider(
    value=100,
    min=0,
    max=2000,
    step=50,
    description=""
)

input_end_widget = widgets.IntSlider(
    value=900,
    min=100,
    max=3000,
    step=50,
    description=""
)

frequency_widget = widgets.FloatSlider(
    value=5.0,
    min=0.5,
    max=30.0,
    step=0.5,
    description=""
)

parameter_noise_widget = widgets.FloatSlider(
    value=0.05,
    min=0.0,
    max=0.3,
    step=0.01,
    description=""
)

input_noise_widget = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=10.0,
    step=0.5,
    description=""
)

connection_probability_widget = widgets.FloatSlider(
    value=0.15,
    min=0.0,
    max=1.0,
    step=0.05,
    description=""
)

weight_scale_widget = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=5.0,
    step=0.1,
    description=""
)

connection_type_widget = widgets.Dropdown(
    options=["excitatory", "inhibitory", "mixed"],
    value="excitatory",
    description=""
)

synaptic_tau_widget = widgets.FloatSlider(
    value=10.0,
    min=1.0,
    max=50.0,
    step=1.0,
    description=""
)

random_seed_widget = widgets.IntSlider(
    value=42,
    min=1,
    max=100,
    step=1,
    description=""
)

bin_size_widget = widgets.IntSlider(
    value=20,
    min=5,
    max=100,
    step=5,
    description=""
)

show_graph_widget = widgets.Checkbox(
    value=True,
    description="Mostrar grafo"
)

show_matrix_widget = widgets.Checkbox(
    value=True,
    description="Mostrar matriz W"
)


def run_gui(_=None):
    """
    Ejecuta simulación al presionar el botón.
    """

    with output:
        clear_output(wait=True)

        results = simulate_connected_izhikevich_network(
            n_neurons=n_neurons_widget.value,
            neuron_type=neuron_type_widget.value,
            total_time=total_time_widget.value,
            dt=dt_widget.value,
            input_type=input_type_widget.value,
            amplitude=amplitude_widget.value,
            input_start=input_start_widget.value,
            input_end=input_end_widget.value,
            frequency=frequency_widget.value,
            parameter_noise=parameter_noise_widget.value,
            input_noise=input_noise_widget.value,
            connection_probability=connection_probability_widget.value,
            weight_scale=weight_scale_widget.value,
            connection_type=connection_type_widget.value,
            synaptic_tau=synaptic_tau_widget.value,
            random_seed=random_seed_widget.value
        )

        W = results["W"]
        spikes = results["spikes"]
        total_time = total_time_widget.value
        n_neurons = n_neurons_widget.value

        total_spikes = int(spikes.sum())
        firing_rate = total_spikes / n_neurons / (total_time / 1000)

        print("Resumen de simulación")
        print("---------------------")
        print(f"Tipo neuronal: {neuron_type_widget.value}")
        print(f"Descripción: {results['neuron_description']}")
        print(f"Número de neuronas: {n_neurons}")
        print(f"Tipo de conexión: {connection_type_widget.value}")
        print(f"Probabilidad de conexión: {connection_probability_widget.value}")
        print(f"Escala de pesos W: {weight_scale_widget.value}")
        print(f"Conexiones activas: {np.count_nonzero(W)}")
        print(f"Spikes totales: {total_spikes}")
        print(f"Firing rate medio: {firing_rate:.2f} Hz")
        print()

        if show_graph_widget.value:
            plot_connectivity_graph(
                W,
                show_labels=n_neurons <= 20
            )

        if show_matrix_widget.value:
            plot_connectivity_matrix(W)

        plot_network_results(
            results,
            bin_size_ms=bin_size_widget.value
        )


run_button = widgets.Button(
    description="Run simulation",
    button_style="success"
)

run_button.on_click(run_gui)


controls = widgets.VBox([
    widgets.HTML("<h2>Connected Izhikevich Network GUI</h2>"),

    described_widget(
        "<b>Tipo neuronal</b><br>Selecciona el patrón base del modelo de Izhikevich. Afecta a todas las neuronas.",
        neuron_type_widget
    ),

    described_widget(
        "<b>Número de neuronas</b><br>Cantidad de neuronas en la red. Limitado entre 2 y 30 para visualizar mejor.",
        n_neurons_widget
    ),

    described_widget(
        "<b>Tipo de input</b><br>Forma del estímulo externo común que recibe toda la red.",
        input_type_widget
    ),

    described_widget(
        "<b>Amplitud del input</b><br>Intensidad del comando externo aplicado a la red.",
        amplitude_widget
    ),

    described_widget(
        "<b>Tiempo total</b><br>Duración de la simulación en milisegundos.",
        total_time_widget
    ),

    described_widget(
        "<b>dt</b><br>Paso temporal de integración. Valores menores son más precisos pero más lentos.",
        dt_widget
    ),

    described_widget(
        "<b>Inicio del input</b><br>Momento en que comienza el estímulo externo.",
        input_start_widget
    ),

    described_widget(
        "<b>Fin del input</b><br>Momento en que termina el estímulo externo.",
        input_end_widget
    ),

    described_widget(
        "<b>Frecuencia</b><br>Solo afecta al input sinusoidal. Controla la oscilación del estímulo.",
        frequency_widget
    ),

    described_widget(
        "<b>Ruido de parámetros</b><br>Introduce pequeñas diferencias internas entre neuronas modificando a, b, c y d.",
        parameter_noise_widget
    ),

    described_widget(
        "<b>Ruido de input</b><br>Agrega variabilidad individual al input recibido por cada neurona.",
        input_noise_widget
    ),

    described_widget(
        "<b>Probabilidad de conexión</b><br>Probabilidad de que exista una conexión entre dos neuronas.",
        connection_probability_widget
    ),

    described_widget(
        "<b>Escala de pesos W</b><br>Controla qué tan fuertes son las conexiones sinápticas entre neuronas.",
        weight_scale_widget
    ),

    described_widget(
        "<b>Tipo de conexión</b><br>Define si la red es excitatoria, inhibitoria o mixta.",
        connection_type_widget
    ),

    described_widget(
        "<b>Tiempo sináptico</b><br>Controla cuánto tarda en decaer el efecto de un spike sobre otras neuronas.",
        synaptic_tau_widget
    ),

    described_widget(
        "<b>Seed</b><br>Fija la aleatoriedad para poder repetir resultados.",
        random_seed_widget
    ),

    described_widget(
        "<b>Bin de actividad</b><br>Tamaño de la ventana temporal para contar spikes poblacionales.",
        bin_size_widget
    ),

    widgets.HBox([show_graph_widget, show_matrix_widget]),
    run_button
])

display(controls, output)

Output()